# 07 — Head-to-Head Coupling (Exploratory)

**Question**: When two teams are tracked simultaneously, is one team's collective order
associated with the opponent's movement, distance compression, or outcome?

This notebook addresses **§3 Head-to-Head Coupling** and **§6 Team Coupling** from the README.

## Analyses
| Analysis | Method |
|----------|--------|
| **A** | Correlation of team A and B polarisation time series |
| **B** | Cross-correlation: does A's order predict B's order with a lag? |
| **C** | Inter-team centroid distance vs order of either team |
| **D** | Coupled hazard: does opponent order modulate termination risk? |
| **E** | Polarisation modes: PCA of joint A+B polarisation state |

**Caveat**: Only head-to-head fixtures have both teams tracked.
Results here are **exploratory** — do not over-interpret.

**Prerequisite**: Run `01_data_loading.ipynb` with `h2h_fixtures` loaded.  
Caches: `df_pmv_teamA`, `df_pmv_teamB`, `hazard_intervals_h2h`.

In [ ]:
import sys, os
from pathlib import Path

_HERE = Path(os.getcwd())
_REPO = _HERE.parents[2]
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as signal
import scipy.stats as stats
from sklearn.decomposition import PCA

from analysis.levy_paper.util.paper_utils import (
    configure_paper_plotting,
    load_cache, save_figure,
    plot_ccdf,
    STATE_COLORS, STATE_LABELS, TEAM_COLORS, A0,
)
configure_paper_plotting()

# Attempt to load head-to-head caches
_MISSING = []
for _name in ["df_pmv_teamA", "df_pmv_teamB"]:
    try:
        globals()[_name] = load_cache(_name)
        print(f"Loaded {_name}: {len(globals()[_name]):,} rows")
    except FileNotFoundError:
        _MISSING.append(_name)
        print(f"Cache not found: {_name}")

if _MISSING:
    print()
    print("To generate head-to-head caches, re-run 01_data_loading.ipynb")
    print("with the h2h_fixtures block enabled and these additional saves:")
    print("  save_cache(df_pmv_A, 'df_pmv_teamA')")
    print("  save_cache(df_pmv_B, 'df_pmv_teamB')")

## A — Simultaneous Polarisation: Team A vs Team B

Align both teams' polarisation time series by match timestamp
and compute Pearson correlation.

In [ ]:
if "df_pmv_teamA" in globals() and "df_pmv_teamB" in globals():
    # Merge on timestamp (inner join — common frames only)
    merged = pd.merge(
        df_pmv_teamA[["timestamp", "match_id", "p_group"]].rename(columns={"p_group": "p_A"}),
        df_pmv_teamB[["timestamp", "match_id", "p_group"]].rename(columns={"p_group": "p_B"}),
        on=["timestamp", "match_id"],
    ).dropna()

    print(f"Aligned frames: {len(merged):,}")

    r_AB, pval_AB = stats.pearsonr(merged["p_A"], merged["p_B"])
    print(f"Pearson r(p_A, p_B) = {r_AB:.3f}  (p={pval_AB:.2e})")

    fig_A, ax_A = plt.subplots(figsize=(5, 4))
    ax_A.hexbin(merged["p_A"], merged["p_B"], gridsize=40, cmap="YlOrRd", mincnt=1)
    ax_A.set_xlabel("Team A polarisation $p_A$")
    ax_A.set_ylabel("Team B polarisation $p_B$")
    ax_A.set_title(f"A — Joint polarisation  ($r$={r_AB:.2f})")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped — caches not available.")

## B — Cross-Correlation: Does Order Lead or Lag?

A positive peak at lag $k > 0$ means Team A's order predicts Team B's order
$k$ seconds later (Team A leads).

In [ ]:
if "merged" in globals():
    p_A = merged["p_A"].values
    p_B = merged["p_B"].values

    # Normalise
    p_A_n = (p_A - p_A.mean()) / (p_A.std() + 1e-9)
    p_B_n = (p_B - p_B.mean()) / (p_B.std() + 1e-9)

    max_lag = 30  # seconds
    xcorr = signal.correlate(p_A_n, p_B_n, mode="full")
    xcorr /= len(p_A)   # normalise
    lags = signal.correlation_lags(len(p_A_n), len(p_B_n), mode="full")

    mask = (lags >= -max_lag) & (lags <= max_lag)

    fig_B, ax_B = plt.subplots(figsize=(6, 3))
    ax_B.plot(lags[mask], xcorr[mask], "k-", lw=1.2)
    ax_B.axvline(0, color="grey", lw=0.8, ls="--")
    ax_B.axhline(0, color="grey", lw=0.5)
    ax_B.set_xlabel("Lag (s)  [+ = A leads]")
    ax_B.set_ylabel("Cross-correlation")
    ax_B.set_title("B — Polarisation cross-correlation (A vs B)")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped.")

## C — Inter-Team Distance vs Order

Does one team's high collective order compress the gap between team centroids?

In [ ]:
traj = load_cache("trajectory_long")

# Extract centroid positions per team per timestamp
if "team" in traj.columns:
    centroid_traj = traj.loc[traj["entity"] == "centroid"][["timestamp", "match_id", "team", "x_m", "y_m"]]
    pivot_c = centroid_traj.pivot_table(index=["timestamp", "match_id"], columns="team",
                                         values=["x_m", "y_m"])
    pivot_c.columns = [f"{col[0]}_{col[1]}" for col in pivot_c.columns]
    pivot_c = pivot_c.dropna().reset_index()

    teams = centroid_traj["team"].unique()
    if len(teams) >= 2:
        tA, tB = teams[0], teams[1]
        pivot_c["dist_AB"] = np.sqrt(
            (pivot_c[f"x_m_{tA}"] - pivot_c[f"x_m_{tB}"]) ** 2 +
            (pivot_c[f"y_m_{tA}"] - pivot_c[f"y_m_{tB}"]) ** 2
        )

        if "merged" in globals():
            merged_dist = pd.merge(pivot_c, merged[["timestamp", "match_id", "p_A", "p_B"]],
                                   on=["timestamp", "match_id"])

            fig_C, axes_C = plt.subplots(1, 2, figsize=(9, 3.5))
            for ax, pvar, label, color in [
                (axes_C[0], "p_A", "Team A", TEAM_COLORS["A"]),
                (axes_C[1], "p_B", "Team B", TEAM_COLORS["B"]),
            ]:
                ax.hexbin(merged_dist[pvar], merged_dist["dist_AB"],
                          gridsize=30, cmap="Blues", mincnt=1)
                r, p = stats.pearsonr(merged_dist[pvar], merged_dist["dist_AB"])
                ax.set_xlabel(f"Polarisation $p_{{{label[-1]}}}$")
                ax.set_ylabel("Inter-centroid distance (m)")
                ax.set_title(f"C — {label}  ($r$={r:.2f})")
            plt.tight_layout()
            plt.show()
else:
    print("No 'team' column in trajectory — skipped.")

## D — Coupled Hazard Model

Does the **opponent's** collective order modulate a team's run termination risk?

Extend the hazard model:
$$h(a, z_{p_A}, z_{p_B}) = h_0(a) \exp(\beta_A z_{p_A} + \beta_B z_{p_B})$$

If $\beta_B > 0$: high opponent order *increases* your hazard (forces faster turns).
If $\beta_B < 0$: high opponent order *protects* your runs (mutual synchrony).

In [ ]:
try:
    haz_h2h = load_cache("hazard_intervals_h2h")
    print(f"H2H hazard intervals: {len(haz_h2h):,}")

    # Fit coupled hazard
    from scipy.optimize import minimize

    df_h = haz_h2h.dropna(subset=["p_focal", "p_opponent", "event", "age_bin"]).copy()
    df_h["z_focal"]    = (df_h["p_focal"]    - df_h["p_focal"].mean())    / df_h["p_focal"].std()
    df_h["z_opponent"] = (df_h["p_opponent"] - df_h["p_opponent"].mean()) / df_h["p_opponent"].std()

    age_arr = df_h["age_bin"].values.astype(float)
    z_f     = df_h["z_focal"].values
    z_o     = df_h["z_opponent"].values
    ev      = df_h["event"].values

    def neg_ll_coupled(theta):
        lam, mu, beta_f, beta_o = theta
        if lam <= 0 or mu <= 0:
            return 1e9
        h0 = lam + mu / (A0 + age_arr)
        h  = h0 * np.exp(beta_f * z_f + beta_o * z_o)
        h  = np.clip(h, 1e-12, None)
        return -np.sum(ev * np.log(h) - h)

    res = minimize(neg_ll_coupled, x0=[0.02, 1.0, -0.3, 0.2], method="Nelder-Mead")
    lam, mu, beta_f, beta_o = res.x
    print(f"Coupled model:")
    print(f"  λ∞ = {lam:.4f}  μ = {mu:.3f}")
    print(f"  β_focal    = {beta_f:.3f}  HR_focal    = {np.exp(beta_f):.3f}")
    print(f"  β_opponent = {beta_o:.3f}  HR_opponent = {np.exp(beta_o):.3f}")

    # Forest plot
    fig_D, ax_D = plt.subplots(figsize=(4.5, 2.5))
    labels = ["Focal order", "Opponent order"]
    hrs    = [np.exp(beta_f), np.exp(beta_o)]
    colors = [TEAM_COLORS["A"], TEAM_COLORS["B"]]
    ax_D.scatter(hrs, [0, 1], color=colors, s=80, zorder=3)
    ax_D.axvline(1.0, color="k", lw=0.8, ls="--")
    ax_D.set_yticks([0, 1])
    ax_D.set_yticklabels(labels)
    ax_D.set_xlabel("Hazard ratio (HR)")
    ax_D.set_title("D — Coupled hazard model")
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print("hazard_intervals_h2h not found — build in notebook 01 from h2h fixtures.")

## E — Polarisation Modes (PCA of Joint A+B State)

**Outstanding question (README §6)**: Do teams share polarisation 'modes'
(collective states that both teams enter simultaneously)?

Apply PCA to the joint $(p_A, p_B)$ time series.
PC1 = in-phase mode (both teams polarised together).
PC2 = anti-phase mode (one high, one low).

In [ ]:
if "merged" in globals():
    X = merged[["p_A", "p_B"]].dropna().values
    pca = PCA(n_components=2)
    pca.fit(X)

    print("PCA explained variance ratio:", pca.explained_variance_ratio_.round(3))
    print("PC1 loadings:", pca.components_[0].round(3),
          "→", "in-phase" if pca.components_[0][0] * pca.components_[0][1] > 0 else "anti-phase")
    print("PC2 loadings:", pca.components_[1].round(3))

    scores = pca.transform(X)

    fig_E, (ax_E1, ax_E2) = plt.subplots(1, 2, figsize=(9, 3.5))

    ax_E1.scatter(X[:, 0], X[:, 1], alpha=0.1, s=5, c=scores[:, 0], cmap="RdBu_r")
    ax_E1.set_xlabel("$p_A$"); ax_E1.set_ylabel("$p_B$")
    ax_E1.set_title("E — Joint polarisation (coloured by PC1)")

    for i, (label, color) in enumerate(zip(["PC1 (in-phase)", "PC2 (anti-phase)"],
                                            ["#1b7837", "#d6604d"])):
        ax_E2.plot(range(min(500, len(scores))), scores[:500, i],
                   color=color, lw=0.8, label=label, alpha=0.8)
    ax_E2.set_xlabel("Time (s)"); ax_E2.set_ylabel("PC score")
    ax_E2.set_title("E — Polarisation modes over time")
    ax_E2.legend(fontsize=8)

    plt.tight_layout()
    plt.show()
else:
    print("Skipped — merged h2h data not available.")